# Feature Engineering Using Scikit-Learn

### Do You Want to Build a Snowman?

Prepare the inputs for a model that predicts snowman height.

<img src="../assets/olaf.jpeg" width="300" alt="Snowman illustration">

This deliberately tiny, fictional dataset is a **preprocessing exercise**, not evidence that lunch or dinner predicts snowman height. Treat all rows as a supplied training sample; do not report a model generalization score from these 14 examples.

**Business connection:** in a housing dataset, the same steps could impute missing floor areas, scale numerical measurements, and encode property types before predicting price. The workflow transfers; the useful features and preprocessing choices depend on the task.

#### Load Packages

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

#### Load Data

In [ ]:
data = {
    "temp": [-3, 5, 0, 7, 3, -1, 1, None, -6, 3, 0, -1, None, -2],
    "lunch": [
        "soup",
        "sandwich",
        "soup",
        "burger",
        "sandwich",
        "soup",
        "cereal",
        "salad",
        "sandwich",
        "burger",
        "soup",
        "cereal",
        "burger",
        "soup",
    ],
    "dinner": [
        "pizza",
        "pizza",
        "noodles",
        None,
        "fishsticks",
        "pizza",
        None,
        "fishsticks",
        "noodles",
        "pizza",
        None,
        "pizza",
        "fishsticks",
        "pizza",
    ],
    "precipitation": [
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "no",
    ],
    "height_snowman_cm": [100, 0, 75, 0, 20, 25, 0, 35, 170, 0, 85, 85, 45, 0],
}

df_train = pd.DataFrame(data=data).fillna(np.nan)
df_train

## One Possible Solution

1. Separate `df_train` into `X_train` and `y_train`; the target is `height_snowman_cm`.
2. Identify numeric, nominal, and binary inputs, then inspect missing values.
3. Impute `temp` with its training median and `dinner` with its most frequent value. Row order has no time meaning here, so do not interpolate between unrelated observations.
4. Scale `temp` and one-hot encode `lunch`, `dinner`, and `precipitation`.
5. Use the tools imported above: `SimpleImputer`, `StandardScaler`, and `OneHotEncoder`. Combine the transformed DataFrames with `pd.concat(axis=1)`.
6. Produce `X_train_fe` with numerical values, no missing values, the same row count and index as `X_train`, and no target column.

These are example choices, not the only valid answer: explain why your transformations suit each variable. Fit on training data only. To transform new rows later, reuse the fitted objects instead of recomputing statistics.

1. **Separate the DataFrame `df_train` into `X_train` and `y_train`**

In [ ]:
target = "height_snowman_cm"

In [ ]:
# Feature matrix
X_train = df_train.drop(target, axis=1)
y_train = df_train[target]

In [ ]:
X_train.columns

2.1 **Identify which variables are binary, categorical and  numeric**

| Column Name | Type     |
|----------|----------------|
| temp       |numeric   |
| lunch        | categorical nominal|
| dinner        | categorical nominal|
| precipitation        | categorical binary|


2.2 **Check which variables have missing values**

In [ ]:
X_train.isna().sum()

In [ ]:
sns.heatmap(data=X_train.isna());

The variables `temp` and `dinner` contain missing values at observations 7 and 12 for `temp`, and at observations 3, 6, and 10 for `dinner`, respectively.

2.3 **Impute missing values**

**Numeric Variable `temp`**

Use a median learned on the training sample. Unlike interpolation across rows, this transformation gives the same result for a new observation regardless of its position in a batch.

In [ ]:
temp_imputer = SimpleImputer(strategy="median").set_output(transform="pandas")

In [ ]:
temp_imputer.fit(X_train[["temp"]])

In [ ]:
temp_imputed_train = temp_imputer.transform(X_train[["temp"]])
temp_imputed_train

In [ ]:
temp_scaler = StandardScaler().set_output(transform="pandas")

In [ ]:
temp_scaled_train = temp_scaler.fit_transform(temp_imputed_train)
temp_scaled_train

**Nominal Variable `dinner`**

We can fill the missing value by using as strategy the most frequent category

In [ ]:
dinner_imputer = SimpleImputer(
    strategy="most_frequent",
).set_output(transform="pandas")

In [ ]:
dinner_imputer.fit(X_train[["dinner"]])

In [ ]:
dinner_imputed_train = dinner_imputer.transform(X_train[["dinner"]])
dinner_imputed_train

2.4 **Encode Categorical variables**

**`lunch` and `precipitation`**

In [ ]:
subset_categoric = ["lunch", "precipitation"]

In [ ]:
ohe_encoder = OneHotEncoder(sparse_output=False, drop="first").set_output(
    transform="pandas"
)

In [ ]:
ohe_encoder.fit(X_train[subset_categoric])

In [ ]:
lunch_prep_encoded_train = ohe_encoder.transform(X_train[subset_categoric])
lunch_prep_encoded_train

**`encoding dinner`**

In [ ]:
dinner_encoder = OneHotEncoder(sparse_output=False, drop="first").set_output(
    transform="pandas"
)

In [ ]:
dinner_encoder.fit(dinner_imputed_train[["dinner"]])

The dinner encoder receives the **imputed dinner column**. The lunch and precipitation encoder handles the other categorical inputs. No feature is duplicated.

This solution uses `drop="first"`, keeping one fewer indicator per categorical input. The advanced solution keeps all indicators, so their output column counts differ intentionally. Both are valid preprocessing choices here. These introductory encoders use the default `handle_unknown="error"`: a new category raises an error rather than silently becoming the reference category.

In [ ]:
dinner_encoded_train = dinner_encoder.transform(dinner_imputed_train[["dinner"]])
dinner_encoded_train

3. **Create X_train_fe**

In [ ]:
transformed_dataframes = [
    temp_scaled_train,
    dinner_encoded_train,
    lunch_prep_encoded_train,
]

In [ ]:
X_train_fe = pd.concat(transformed_dataframes, axis=1)
assert X_train_fe.index.equals(y_train.index)
assert not X_train_fe.isna().any().any()
X_train_fe

### Why These Choices?

Median imputation is less sensitive to extreme temperatures than the mean. The dinner mode reuses an existing category, but an explicit missing category is another option. Standard scaling puts temperature on a standardized scale; one-hot encoding avoids inventing an order for meals. These are preprocessing examples, not validated evidence of predictive usefulness.

### Reuse the Fitted Objects

The known-category row should transform successfully. The second row deliberately demonstrates the default encoder error for `wrap`. We catch only that expected error and display it; we do not suppress warnings or refit the encoder.

In [ ]:
# New inputs only: the fitted transformers must not learn from these rows.
X_new = pd.DataFrame(
    {
        "temp": [np.nan, 2.0],
        "lunch": ["soup", "wrap"],
        "dinner": [np.nan, "pizza"],
        "precipitation": ["yes", "no"],
    },
    index=["known", "new_category"],
)
X_new

In [ ]:
known = X_new.loc[["known"]]
known_temp = temp_scaler.transform(temp_imputer.transform(known[["temp"]]))
known_dinner = dinner_encoder.transform(dinner_imputer.transform(known[["dinner"]]))
known_other = ohe_encoder.transform(known[subset_categoric])
known_fe = pd.concat([known_temp, known_dinner, known_other], axis=1)
assert known_fe.columns.equals(X_train_fe.columns)
assert known_fe.index.equals(known.index)
assert not known_fe.isna().any().any()
print(known_fe)

try:
    ohe_encoder.transform(X_new.loc[["new_category"], subset_categoric])
except ValueError as error:
    if "unknown categories" not in str(error):
        raise
    print(f"Expected unseen-category error: {error}")

### Short Answers

1. `transform` reuses training statistics. Refitting on test data lets the evaluation data influence preprocessing and changes the representation expected by the fitted model.
2. Meal names have no natural order. Codes such as 0, 1, 2 introduce ordering and numerical distances that those models can interpret as meaningful. One-hot encoding avoids this assumption.
3. No. `RobustScaler` centers by the training median and scales by the training IQR. Those statistics are less sensitive to extremes, but the observations remain in the data.